# Preprocessing Data Spesifikasi Laptop Gaming (Persiapan Model Regresi)

**Aplikasi:** GamerGear - Web Platform Rekomendasi Hardware & Laptop Gaming  
**Tujuan:** Mempersiapkan dataset mentah spesifikasi laptop menjadi data bersih, terstandarisasi, dan terdistribusi normal yang siap dilatih menggunakan model regresi machine learning untuk estimasi/prediksi harga laptop gaming.  

---

## 📋 Ringkasan Alur Kerja Preprocessing:
1. **Langkah 1: Standardisasi Satuan**
   - Konversi harga mata uang asing (Rupee India / ₹) ke Rupiah (IDR) menggunakan kurs tetap yang didokumentasikan secara transparan.
   - Penyeragaman kapasitas RAM dan SSD/Storage ke dalam satuan seragam: **Gigabyte (GB)**.
2. **Langkah 2: Ekstraksi Fitur Teks Spesifikasi**
   - Ekstraksi seri GPU (contoh: RTX 4060, RTX 3050, RTX 5070) menggunakan Regular Expression (`re`) dan metode `str.extract()` pandas.
   - Ekstraksi kelas dan generasi CPU (Core i5, Core i7, Core i9, Ryzen 5, Ryzen 7, Core Ultra 7/9, dll.).
   - Ekstraksi ukuran layar (*inch*) dan refresh rate layar (*Hz*).
3. **Langkah 3: Pemetaan Tier Performa (Ordinal Encoding)**
   - Mengubah data kategori GPU dan CPU menjadi tingkatan performa ordinal bertingkat (*Tier 1 s/d Tier 6*).
   - Mempertahankan hierarki komputasi hardware agar model regresi dapat mengenali urutan kekuatan komponen.
   - Menggunakan kamus manual (`pandas.map()`) serta `scikit-learn.preprocessing.OrdinalEncoder`.
4. **Langkah 4: Penanganan Nilai Hilang (Missing Values) & Pencilan (Outliers)**
   - Imputasi nilai kosong (`fillna()` dan `SimpleImputer`).
   - Penanganan data anomali / tidak masuk akal (seperti kapasitas SSD 1 GB / 8 GB pada laptop ASUS ROG Strix Scar bernilai tinggi).
   - Analisis pencilan harga dengan metode Interquartile Range (IQR).
5. **Langkah 5: Transformasi Distribusi Harga**
   - Evaluasi kemencengan (*skewness*) harga.
   - Menerapkan transformasi logaritma (`np.log1p`) agar distribusi harga mendekati normal untuk memenuhi asumsi model regresi.

## 0. Inisialisasi & Import Library

In [1]:
import os
import re
import numpy as np
import pandas as pd

# Opsi konfigurasi tampilan pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Pengecekan scikit-learn untuk OrdinalEncoder & SimpleImputer
try:
    from sklearn.preprocessing import OrdinalEncoder
    from sklearn.impute import SimpleImputer
    SKLEARN_AVAILABLE = True
    print("✅ scikit-learn siap digunakan.")
except ImportError:
    SKLEARN_AVAILABLE = False
    print("ℹ️ scikit-learn belum terpasang di venv ini. Fallback native pandas & numpy akan digunakan.")

# Pengecekan library visualisasi
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    PLT_AVAILABLE = True
    print("✅ matplotlib dan seaborn siap digunakan.")
except ImportError:
    PLT_AVAILABLE = False
    print("ℹ️ matplotlib/seaborn belum terpasang. Visualisasi diagram grafik dilewati.")

✅ scikit-learn siap digunakan.
ℹ️ matplotlib/seaborn belum terpasang. Visualisasi diagram grafik dilewati.


### 0.1 Memuat Dataset Mentah Spesifikasi Laptop
Dataset mentah dibaca dari folder `raw-dataset/final_cleaned_dataset.csv`.

In [2]:
# Menentukan path relatif dataset terhadap posisi notebook
data_path = os.path.join('..', 'raw-dataset', 'final_cleaned_dataset.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('raw-dataset', 'final_cleaned_dataset.csv')

df = pd.read_csv(data_path)
print(f"Dataset berhasil dimuat! Dimensi: {df.shape[0]} baris dan {df.shape[1]} kolom.\n")
display(df.head(5))
print("\nInformasi tipe data dan ringkasan kolom:")
df.info()

Dataset berhasil dimuat! Dimensi: 346 baris dan 16 kolom.



,name,specs_score,price,processor,graphics_card,os,warranty,VRAM(GB),cores,thread,ram(GB),ram_version,storage_size,storage_type,display_size,display_resolution
0,Lenovo LOQ 83DV007GIN Gaming Laptop,59,132000,13th Gen Intel Core i5 13450HX,6 GB NVIDIA GeForce RTX 4050,11,1,6,10 (6P + 4E),16,16,DDR5,512 GB,SSD,15.6,1920 x 1080
1,HP Victus 15-fb3125AX Gaming Laptop,50,82494,7th Gen AMD Ryzen 7 7445H,NVIDIA GeForce RTX 4050,11,1,6,6,12,16,DDR5,512 GB,SSD,15.6,1920 x 1080
2,HP Victus 15-fa2192TX Gaming Laptop,53,78990,13th Gen Intel Core i5 13420H,4 GB NVIDIA GeForce RTX 3050,11,1,4,8,12,16,DDR4,512 GB,SSD,15.6,1920 x 1080
3,Lenovo LOQ Essential 83S000JCIN Gaming Laptop,64,89990,7th Gen AMD Ryzen 5 7535HS,6 GB NVIDIA GeForce RTX 3050,11,1,6,6,12,16,DDR5,512 GB,SSD,15.6,1920 x 1080
4,Lenovo LOQ 2023 Gaming Laptop,54,89999,13th Gen Intel Core i5 13420H,8 GB NVIDIA GeForce RTX 4060,11,1,8,8,12,8,DDR5,512 GB,SSD,15.6,1920 x 1080



Informasi tipe data dan ringkasan kolom:
<class 'pandas.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name                346 non-null    str    
 1   specs_score         346 non-null    int64  
 2   price               346 non-null    int64  
 3   processor           346 non-null    str    
 4   graphics_card       346 non-null    str    
 5   os                  346 non-null    int64  
 6   warranty            346 non-null    int64  
 7   VRAM(GB)            346 non-null    int64  
 8   cores               346 non-null    str    
 9   thread              346 non-null    int64  
 10  ram(GB)             346 non-null    int64  
 11  ram_version         346 non-null    str    
 12  storage_size        346 non-null    str    
 13  storage_type        346 non-null    str    
 14  display_size        346 non-null    float64
 15  display_resolution  346 no

---
## 1. Langkah 1: Standardisasi Satuan

### 1.1 Dokumentasi Kurs Tetap (Mata Uang Asing ke IDR)
> **Catatan Dokumentasi Kurs Tetap:**
> - **Mata Uang Asal:** Rupee India (INR / ₹) dari hasil ekstraksi data portal e-commerce teknologi.
> - **Mata Uang Target:** Rupiah Indonesia (IDR / Rp).
> - **Nilai Kurs Acuan Tetap:** **1 INR = Rp 190,00 (IDR)**  
> - *Justifikasi:* Nilai tukar rata-rata Rupiah terhadap Rupee India per November 2024–2026 berada pada rentang Rp 185 – Rp 192 per 1 INR. Menetapkan konstanta `190.0` memastikan konsistensi konversi harga dan keterulangan (*reproducibility*) hasil pemodelan regresi.

### 1.2 Penyeragaman Satuan Kapasitas (RAM & SSD/Storage ke GB)
> - **RAM:** Kolom `ram(GB)` dipastikan bertipe numerik integer dalam satuan Gigabyte.
> - **Storage / SSD:** Kolom `storage_size` masih bertipe teks campuran dengan satuan bervariasi (`512 GB`, `1 TB`, `2 TB`, `4 TB`, dll.).
> - Nilai Terabyte (TB) dikonversi ke Gigabyte menggunakan standar biner komputer ($1 \text{ TB} = 1024 \text{ GB}$).

In [3]:
# Tetapkan konstanta kurs tetap
KURS_INR_TO_IDR = 190.0

# 1. Konversi Harga ke Rupiah (IDR) menggunakan operasi perkalian kolom pandas
df['price_idr'] = df['price'] * KURS_INR_TO_IDR

# 2. Standardisasi Kapasitas RAM ke integer GB
df['ram_gb'] = df['ram(GB)'].astype(int)

# 3. Standardisasi Kapasitas Storage / SSD ke satuan GB
def standardize_storage_to_gb(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.search(r'(\d+(?:\.\d+)?)\s*(GB|TB)', text, re.IGNORECASE)
    if match:
        capacity = float(match.group(1))
        unit = match.group(2).upper()
        if unit == 'TB':
            return int(capacity * 1024) # 1 TB = 1024 GB
        elif unit == 'GB':
            return int(capacity)
    return np.nan

df['storage_gb'] = df['storage_size'].apply(standardize_storage_to_gb)

print("=== REKAPITULASI HASIL STANDARDISASI SATUAN ===")
print(f"Distribusi Kapasitas Storage Baru (GB):\n{df['storage_gb'].value_counts().sort_index()}\n")
display(df[['name', 'price', 'price_idr', 'ram(GB)', 'ram_gb', 'storage_size', 'storage_gb']].head(5))

=== REKAPITULASI HASIL STANDARDISASI SATUAN ===
Distribusi Kapasitas Storage Baru (GB):
storage_gb
1         1
8         1
256       1
512     191
1024    131
2048     19
4096      2
Name: count, dtype: int64



,name,price,price_idr,ram(GB),ram_gb,storage_size,storage_gb
0,Lenovo LOQ 83DV007GIN Gaming Laptop,132000,25080000.0,16,16,512 GB,512
1,HP Victus 15-fb3125AX Gaming Laptop,82494,15673860.0,16,16,512 GB,512
2,HP Victus 15-fa2192TX Gaming Laptop,78990,15008100.0,16,16,512 GB,512
3,Lenovo LOQ Essential 83S000JCIN Gaming Laptop,89990,17098100.0,16,16,512 GB,512
4,Lenovo LOQ 2023 Gaming Laptop,89999,17099810.0,8,8,512 GB,512


---
## 2. Langkah 2: Ekstraksi Fitur Teks Spesifikasi

Data teks mentah pada kolom `graphics_card`, `processor`, dan `name` mengandung informasi kunci spesifikasi yang belum terstruktur.
Pada langkah ini, kita memanfaatkan modul `re` (Regular Expression) dan metode `.str.extract()` / `.apply()` pada pandas untuk mengekstrak:

1. **Seri GPU (`gpu_series`):** Mengambil kode kartu grafis seperti `RTX 4060`, `RTX 3050`, `RTX 5070`, `RX 6700S`, `Intel Iris Xe`, dll.
2. **Kelas & Generasi CPU (`cpu_class`, `cpu_gen`, `cpu_brand`):** Memisahkan keluarga prosesor (Core i5, Core i7, Core i9, Core Ultra, Ryzen 5, Ryzen 7) dan nomor generasinya (Generasi 11, 12, 13, 14 Intel, atau Generasi 6, 7, 8 AMD).
3. **Ukuran Layar (`screen_size_inch`):** Mengambil ukuran diagonal layar (contoh: 15.6, 16.0, 14.0 inch) dan menyelaraskan dengan kolom display.
4. **Refresh Rate Layar (`refresh_rate_hz`):** Mengambil angka refresh rate dalam Hertz (contoh: 144Hz, 165Hz, 240Hz).

In [4]:
# 2.1 Ekstraksi Seri GPU dari Kolom graphics_card
def extract_gpu_series(text):
    if pd.isna(text):
        return 'Unknown'
    text = str(text)
    # Deteksi pola NVIDIA RTX
    rtx_match = re.search(r'(RTX\s*\d{4}(?:\s*(?:Ti|TI|Super|SUPER|A))?)', text, re.IGNORECASE)
    if rtx_match:
        val = re.sub(r'\s+', ' ', rtx_match.group(1).upper().strip())
        val = val.replace('TI', 'Ti').replace('SUPER', 'Super')
        val = re.sub(r'(\d{4})Ti', r'\1 Ti', val)
        return val
    # Deteksi pola NVIDIA GTX
    gtx_match = re.search(r'(GTX\s*\d{4}(?:\s*(?:Ti|TI))?)', text, re.IGNORECASE)
    if gtx_match:
        val = re.sub(r'\s+', ' ', gtx_match.group(1).upper().strip())
        val = val.replace('TI', 'Ti')
        val = re.sub(r'(\d{4})Ti', r'\1 Ti', val)
        return val
    # Deteksi pola AMD Radeon RX
    rx_match = re.search(r'(RX\s*\d{4}[A-Z]?)', text, re.IGNORECASE)
    if rx_match:
        return re.sub(r'\s+', ' ', rx_match.group(1).upper().strip())
    # Deteksi Integrated GPU
    if re.search(r'Iris\s*Xe', text, re.I):
        return 'Intel Iris Xe'
    if re.search(r'Intel\s*Arc', text, re.I):
        return 'Intel Arc'
    if re.search(r'Radeon', text, re.I):
        return 'AMD Radeon Graphics'
    return 'Integrated/Other'

df['gpu_series'] = df['graphics_card'].apply(extract_gpu_series)
print("=== HASIL EKSTRAKSI SERI GPU (TOP 10) ===")
print(df['gpu_series'].value_counts().head(10))

=== HASIL EKSTRAKSI SERI GPU (TOP 10) ===
gpu_series
RTX 3050       97
RTX 4050       59
RTX 5050       38
RTX 5060       35
RTX 5070       18
RTX 2050       15
RTX 4060       13
RTX 5080        9
RTX 5090        8
RTX 5070 Ti     8
Name: count, dtype: int64


In [5]:
# 2.2 Ekstraksi Brand, Kelas, dan Generasi CPU dari Kolom processor
def extract_cpu_details(text):
    if pd.isna(text):
        return pd.Series({'cpu_brand': 'Unknown', 'cpu_class': 'Other', 'cpu_gen': np.nan})
    text = str(text)
    
    # Deteksi Brand Pabrikan
    brand = 'Other'
    if re.search(r'Intel|Core', text, re.I):
        brand = 'Intel'
    elif re.search(r'AMD|Ryzen', text, re.I):
        brand = 'AMD'
    elif re.search(r'Snapdragon', text, re.I):
        brand = 'Qualcomm'
        
    # Deteksi Kelas / Keluarga CPU
    cpu_class = 'Other'
    if re.search(r'Ultra\s*9', text, re.I):
        cpu_class = 'Core Ultra 9'
    elif re.search(r'Ultra\s*7', text, re.I):
        cpu_class = 'Core Ultra 7'
    elif re.search(r'i9|Core\s*9', text, re.I):
        cpu_class = 'Core i9'
    elif re.search(r'i7|Core\s*7', text, re.I):
        cpu_class = 'Core i7'
    elif re.search(r'i5|Core\s*5', text, re.I):
        cpu_class = 'Core i5'
    elif re.search(r'i3|Core\s*3', text, re.I):
        cpu_class = 'Core i3'
    elif re.search(r'Ryzen\s*9|Ryzen\s*AI\s*9', text, re.I):
        cpu_class = 'Ryzen 9'
    elif re.search(r'Ryzen\s*7|Ryzen\s*AI\s*7', text, re.I):
        cpu_class = 'Ryzen 7'
    elif re.search(r'Ryzen\s*5', text, re.I):
        cpu_class = 'Ryzen 5'
    elif re.search(r'Ryzen\s*3', text, re.I):
        cpu_class = 'Ryzen 3'
        
    # Deteksi Nomor Generasi CPU
    gen_match = re.search(r'(\d+)(?:th|nd|rd|st)\s*Gen', text, re.I)
    if gen_match:
        gen = int(gen_match.group(1))
    else:
        # Deteksi dari nomor model Intel (e.g., 13450 -> Gen 13, 14900 -> Gen 14, 12450 -> Gen 12)
        intel_model = re.search(r'(?:i[3579]-|Core\s*[3579]\s*)?(\d{2})\d{3}', text)
        if intel_model and brand == 'Intel':
            gen = int(intel_model.group(1))
        else:
            # Deteksi dari nomor model AMD (e.g., 7445 -> Gen 7, 8845 -> Gen 8, 5800 -> Gen 5)
            amd_model = re.search(r'Ryzen\s*[3579]\s*(\d)\d{3}', text)
            if amd_model and brand == 'AMD':
                gen = int(amd_model.group(1))
            else:
                gen = np.nan

    return pd.Series({'cpu_brand': brand, 'cpu_class': cpu_class, 'cpu_gen': gen})

cpu_details = df['processor'].apply(extract_cpu_details)
df[['cpu_brand', 'cpu_class', 'cpu_gen']] = cpu_details

print("=== HASIL EKSTRAKSI KELAS CPU ===")
print(df['cpu_class'].value_counts())
print("\nDistribusi Generasi CPU:")
print(df['cpu_gen'].value_counts().sort_index())

=== HASIL EKSTRAKSI KELAS CPU ===
cpu_class
Core i5         96
Core i7         81
Ryzen 7         62
Core Ultra 9    34
Core i9         18
Ryzen 9         17
Other           16
Ryzen 5         14
Core Ultra 7     8
Name: count, dtype: int64

Distribusi Generasi CPU:
cpu_gen
5.0      5
6.0      4
7.0     37
8.0     11
9.0      4
10.0     1
11.0     4
12.0    30
13.0    73
14.0    52
Name: count, dtype: int64


In [6]:
# 2.3 Ekstraksi Ukuran Layar & 2.4 Ekstraksi Refresh Rate Layar
# Ekstraksi ukuran layar standar laptop (13 - 18 inch) dari nama produk menggunakan str.extract(), fallback ke display_size
screen_extracted = df['name'].str.extract(r'\b(1[3-8](?:\.[0-9])?)\s*(?:inch|-inch)?\b', flags=re.I)[0].astype(float)
df['screen_size_inch'] = screen_extracted.fillna(df['display_size'])

# Ekstraksi Refresh Rate Layar (Hz) dari judul atau teks spesifikasi mentah
df['refresh_rate_hz'] = df['name'].str.extract(r'(\d{2,3})\s*[Hh][Zz]', expand=False).astype(float)

print(f"Rentang Ukuran Layar: {sorted(df['screen_size_inch'].dropna().unique())} inch")
print(f"Laptop dengan refresh rate eksplisit: {df['refresh_rate_hz'].notnull().sum()} unit.")
print(f"Laptop dengan refresh rate NaN: {df['refresh_rate_hz'].isnull().sum()} unit (akan diimputasi pada Langkah 4).\n")
display(df[['name', 'screen_size_inch', 'refresh_rate_hz', 'gpu_series', 'cpu_class', 'cpu_gen']].head(5))

Rentang Ukuran Layar: [np.float64(13.4), np.float64(14.0), np.float64(15.0), np.float64(15.3), np.float64(15.6), np.float64(16.0), np.float64(16.1), np.float64(17.3), np.float64(18.0)] inch
Laptop dengan refresh rate eksplisit: 0 unit.
Laptop dengan refresh rate NaN: 346 unit (akan diimputasi pada Langkah 4).



,name,screen_size_inch,refresh_rate_hz,gpu_series,cpu_class,cpu_gen
0,Lenovo LOQ 83DV007GIN Gaming Laptop,15.6,NaN,RTX 4050,Core i5,13.0
1,HP Victus 15-fb3125AX Gaming Laptop,15.0,NaN,RTX 4050,Ryzen 7,7.0
2,HP Victus 15-fa2192TX Gaming Laptop,15.0,NaN,RTX 3050,Core i5,13.0
3,Lenovo LOQ Essential 83S000JCIN Gaming Laptop,15.6,NaN,RTX 3050,Ryzen 5,7.0
4,Lenovo LOQ 2023 Gaming Laptop,15.6,NaN,RTX 4060,Core i5,13.0


---
## 3. Langkah 3: Pemetaan Tier Performa (Ordinal Encoding)

Dalam model regresi (seperti *Linear Regression*, *Ridge*, *Lasso*, *Random Forest*), komponen hardware seperti GPU dan CPU memiliki hubungan **monotonik bertingkat (ordinal)** dengan harga dan performa.
Jika menggunakan *One-Hot Encoding*, dimensi fitur akan meledak (*curse of dimensionality*) dan model kehilangan informasi urutan kekuatan komputasi.

### 3.1 Skema Tingkatan Tier GPU
- **Tier 1 (Entry / Integrated):** Intel Iris Xe, AMD Radeon Graphics, GTX 1050 Ti, GTX 1650, RX 5500M.
- **Tier 2 (Budget Gaming):** RTX 2050, RTX 3030, RTX 3050 (4GB/6GB), RTX 3050 Ti, GTX 1660 Ti.
- **Tier 3 (Mainstream Gaming):** RTX 3060, RTX 4050, RX 6700S.
- **Tier 4 (High Gaming):** RTX 3070, RTX 3070 Ti, RTX 4060, RTX 5050.
- **Tier 5 (Enthusiast Gaming):** RTX 3080, RTX 4070, RTX 5060, RTX 5070, RTX 5070 Ti.
- **Tier 6 (Flagship / Extreme):** RTX 4080, RTX 4090, RTX 5080, RTX 5090.

### 3.2 Skema Tingkatan Tier CPU
- **Tier 1 (Entry):** Core i3, Ryzen 3, Other.
- **Tier 2 (Mainstream Gaming):** Core i5, Ryzen 5.
- **Tier 3 (High Gaming):** Core i7, Ryzen 7.
- **Tier 4 (High Performance / AI Creator):** Core Ultra 7.
- **Tier 5 (Extreme Flagship):** Core i9, Ryzen 9, Core Ultra 9.

In [7]:
# Metode 1: Kamus Pemetaan Manual (Python Dictionary + pandas .map())
gpu_tier_map = {
    'Intel Iris Xe': 1, 'AMD Radeon Graphics': 1, 'Integrated/Other': 1,
    'GTX 1050 Ti': 1, 'GTX 1650': 1, 'RX 5500M': 1, 'RX 6500M': 1,
    'RTX 2050': 2, 'RTX 3030': 2, 'RTX 3050': 2, 'RTX 3050 A': 2, 'RTX 3050 Ti': 2,
    'GTX 1660 Ti': 2, 'GTX 1660Ti': 2,
    'RTX 2060': 3, 'RTX 3060': 3, 'RTX 4050': 3, 'RX 6700S': 3,
    'RTX 4060': 4, 'RTX 5050': 4, 'RTX 3070': 4, 'RTX 3070 Ti': 4,
    'RTX 4070': 5, 'RTX 5060': 5, 'RTX 5070': 5, 'RTX 5070 Ti': 5, 'RTX 3080': 5, 'RTX 3080 Ti': 5,
    'RTX 4080': 6, 'RTX 4090': 6, 'RTX 5080': 6, 'RTX 5090': 6
}

cpu_tier_map = {
    'Other': 1, 'Core i3': 1, 'Ryzen 3': 1,
    'Core i5': 2, 'Ryzen 5': 2,
    'Core i7': 3, 'Ryzen 7': 3,
    'Core Ultra 7': 4,
    'Core i9': 5, 'Ryzen 9': 5, 'Core Ultra 9': 5
}

df['gpu_tier'] = df['gpu_series'].map(gpu_tier_map).fillna(2).astype(int)
df['cpu_tier'] = df['cpu_class'].map(cpu_tier_map).fillna(2).astype(int)

print("=== DISTRIBUSI TIER GPU ===")
print(df['gpu_tier'].value_counts().sort_index())

print("\n=== DISTRIBUSI TIER CPU ===")
print(df['cpu_tier'].value_counts().sort_index())

=== DISTRIBUSI TIER GPU ===
gpu_tier
1     14
2    125
3     63
4     53
5     68
6     23
Name: count, dtype: int64

=== DISTRIBUSI TIER CPU ===
cpu_tier
1     16
2    110
3    143
4      8
5     69
Name: count, dtype: int64


In [8]:
# Metode 2: scikit-learn OrdinalEncoder (dengan urutan kategori eksplisit)
if SKLEARN_AVAILABLE:
    cpu_categories = ['Other', 'Core i3', 'Core i5', 'Core i7', 'Core Ultra 7', 'Core i9']
    # Harmonisasi kelas CPU ke dalam daftar kategori terurut
    df_cpu_temp = df['cpu_class'].copy().replace({
        'Ryzen 3': 'Core i3', 'Ryzen 5': 'Core i5', 'Ryzen 7': 'Core i7',
        'Ryzen 9': 'Core i9', 'Core Ultra 9': 'Core i9'
    })
    df_cpu_temp = df_cpu_temp.where(df_cpu_temp.isin(cpu_categories), 'Other')
    
    encoder = OrdinalEncoder(categories=[cpu_categories])
    df['cpu_tier_sklearn'] = encoder.fit_transform(df_cpu_temp.values.reshape(-1, 1)).astype(int) + 1
    print("✅ OrdinalEncoder Scikit-learn berhasil diterapkan secara konsisten.")
    display(df[['name', 'cpu_class', 'cpu_tier', 'cpu_tier_sklearn']].head(5))
else:
    print("ℹ️ Menggunakan pemetaan kamus pandas .map() sebagai representasi ordinal utama.")

✅ OrdinalEncoder Scikit-learn berhasil diterapkan secara konsisten.


,name,cpu_class,cpu_tier,cpu_tier_sklearn
0,Lenovo LOQ 83DV007GIN Gaming Laptop,Core i5,2,3
1,HP Victus 15-fb3125AX Gaming Laptop,Ryzen 7,3,4
2,HP Victus 15-fa2192TX Gaming Laptop,Core i5,2,3
3,Lenovo LOQ Essential 83S000JCIN Gaming Laptop,Ryzen 5,2,3
4,Lenovo LOQ 2023 Gaming Laptop,Core i5,2,3


---
## 4. Langkah 4: Penanganan Nilai Hilang (Missing Values) & Pencilan (Outliers)

### 4.1 Identifikasi & Perbaikan Anomali Data Tidak Masuk Akal
Pada analisis eksploratif, terdeteksi anomali nyata pada dataset mentah:
1. **Anomali Storage (<= 8 GB):** Laptop premium *ASUS ROG Strix Scar 18* bernilai ratusan juta rupiah (Rp 90.000.000 s/d Rp 150.000.000) dengan RAM 128 GB dan RTX 5090 tercatat memiliki kapasitas penyimpanan `'1 GB'` dan `'8 GB'`. Hal ini merupakan kesalahan ketik (*typo*) saat proses scraping/input data (seharusnya 1 TB atau 8 TB). Nilai ini kita perbaiki menjadi nilai logis minimum laptop flagship yaitu 1024 GB (1 TB).
2. **Anomali Processor:** Sejumlah baris data mencatat nama GPU (seperti `Intel Iris Xe Graphics` atau `AMD Radeon`) di kolom `processor`.

In [9]:
# 1. Deteksi & Perbaikan Anomali Storage
anomaly_storage = df[df['storage_gb'] <= 8]
print(f"Ditemukan {len(anomaly_storage)} baris dengan anomali kapasitas storage <= 8 GB:")
display(anomaly_storage[['name', 'price_idr', 'processor', 'graphics_card', 'storage_size', 'storage_gb']])

# Koreksi anomali storage menjadi 1024 GB (1 TB SSD)
df.loc[df['storage_gb'] <= 8, 'storage_gb'] = 1024
print(f"✅ Anomali kapasitas storage berhasil dikoreksi ke nilai wajar: 1024 GB (1 TB).")

Ditemukan 2 baris dengan anomali kapasitas storage <= 8 GB:


,name,price_idr,processor,graphics_card,storage_size,storage_gb
21,Asus ROG Strix SCAR 18 2026 Gaming Laptop,151999810.0,Intel Core Ultra 9 290HX Plus,24 GB NVIDIA GeForce RTX 5090,8 GB,8
166,Asus ROG Strix Scar 18,93098100.0,Intel Core Ultra 9 290HX Plus,16 GB NVIDIA GeForce RTX 5080,1 GB,1


✅ Anomali kapasitas storage berhasil dikoreksi ke nilai wajar: 1024 GB (1 TB).


In [10]:
# 2. Penanganan Nilai Hilang (Missing Values) & Imputasi
print("Jumlah nilai hilang (NaN) sebelum imputasi:")
print(df[['price_idr', 'ram_gb', 'storage_gb', 'screen_size_inch', 'refresh_rate_hz', 'gpu_tier', 'cpu_tier']].isnull().sum())

# Imputasi Refresh Rate Layar:
# Nilai median refresh rate standar pada segmen laptop gaming modern adalah 144 Hz
MEDIAN_REFRESH_RATE = 144.0

# Imputasi menggunakan pandas fillna()
df['refresh_rate_hz'] = df['refresh_rate_hz'].fillna(MEDIAN_REFRESH_RATE)

# Demonstrasi imputasi menggunakan scikit-learn SimpleImputer (jika terpasang)
if SKLEARN_AVAILABLE:
    imputer = SimpleImputer(strategy='median')
    df['screen_size_inch'] = imputer.fit_transform(df[['screen_size_inch']]).ravel()

print("\nJumlah nilai hilang (NaN) setelah imputasi:")
print(df[['price_idr', 'ram_gb', 'storage_gb', 'screen_size_inch', 'refresh_rate_hz', 'gpu_tier', 'cpu_tier']].isnull().sum())

Jumlah nilai hilang (NaN) sebelum imputasi:
price_idr             0
ram_gb                0
storage_gb            0
screen_size_inch      0
refresh_rate_hz     346
gpu_tier              0
cpu_tier              0
dtype: int64

Jumlah nilai hilang (NaN) setelah imputasi:
price_idr           0
ram_gb              0
storage_gb          0
screen_size_inch    0
refresh_rate_hz     0
gpu_tier            0
cpu_tier            0
dtype: int64


In [11]:
# 3. Analisis Pencilan Statistik (Outliers) Menggunakan Metode Interquartile Range (IQR)
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return lower_bound, upper_bound, outliers

lb_price, ub_price, price_outliers = detect_outliers_iqr(df['price_idr'])
print(f"Ambang Batas IQR Harga: Bawah = Rp {lb_price:,.0f}, Atas = Rp {ub_price:,.0f}")
print(f"Jumlah outlier harga: {len(price_outliers)} dari {len(df)} data ({len(price_outliers)/len(df)*100:.1f}%)\n")

print("Contoh data outlier harga tinggi:")
display(df.loc[price_outliers.index, ['name', 'price_idr', 'gpu_series', 'gpu_tier', 'ram_gb', 'storage_gb']].head(5))

print("\n💡 Kesimpulan Analisis Outlier:")
print("- Outlier harga di atas BUKAN kesalahan pengetikan/error data, melainkan laptop ultra enthusiast (RTX 4090/5090, RAM 64GB-128GB) yang valid secara pasar.")
print("- Menghapus baris ini akan menghilangkan informasi laptop kelas tertinggi.")
print("- Solusi ideal untuk regresi: Menerapkan Transformasi Logaritma pada Langkah 5 untuk merapatkan sebaran nilai ekstrim!")

Ambang Batas IQR Harga: Bawah = Rp -6,176,900, Atas = Rp 64,503,100
Jumlah outlier harga: 35 dari 346 data (10.1%)

Contoh data outlier harga tinggi:


,name,price_idr,gpu_series,gpu_tier,ram_gb,storage_gb
21,Asus ROG Strix SCAR 18 2026 Gaming Laptop,151999810.0,RTX 5090,6,128,1024
27,MSI Titan 18 HX AI A2XWIG Gaming Laptop,94999810.0,RTX 5080,6,96,4096
40,Asus ROG Zephyrus Duo 16 OLED,110198100.0,RTX 5070 Ti,5,32,2048
45,Asus ROG Zephyrus Duo 16 OLED,138698100.0,RTX 5090,6,64,2048
61,Asus ROG Zephyrus G16 2026 GU605CP-QR117WS Gaming Laptop,70298100.0,RTX 5070,5,32,1024



💡 Kesimpulan Analisis Outlier:
- Outlier harga di atas BUKAN kesalahan pengetikan/error data, melainkan laptop ultra enthusiast (RTX 4090/5090, RAM 64GB-128GB) yang valid secara pasar.
- Menghapus baris ini akan menghilangkan informasi laptop kelas tertinggi.
- Solusi ideal untuk regresi: Menerapkan Transformasi Logaritma pada Langkah 5 untuk merapatkan sebaran nilai ekstrim!


---
## 5. Langkah 5: Transformasi Distribusi Harga (Log Transformation)

Salah satu asumsi fundamental dalam pemodelan regresi (*Ordinary Least Squares*, *Ridge*, *Lasso*, *Linear Regression*) adalah **normalitas residual** dan **homoskedastisitas** (varians error konstan).
Data harga sering memiliki kemencengan positif yang tinggi (*right-skewed / positive-skewed*), di mana sebagian besar data terkonsentrasi di harga murah-menengah, sementara sebagian kecil produk flagship memiliki harga sangat tinggi.

### Formula Transformasi:
$$\text{log\_price\_idr} = \ln(\text{price\_idr} + 1) = \text{np.log1p}(\text{df['price\_idr']})$$

Fungsi `np.log1p()` dari numpy lebih stabil secara numerik karena menghitung $\ln(1 + x)$ dan mencegah terjadinya *undefined* saat nilai mendekati nol.

In [12]:
# Hitung skewness sebelum transformasi
skew_before = df['price_idr'].skew()

# Terapkan transformasi logaritma menggunakan numpy np.log1p
df['log_price_idr'] = np.log1p(df['price_idr'])

# Hitung skewness sesudah transformasi
skew_after = df['log_price_idr'].skew()

print("=== PERBANDINGAN DISTRIBUSI SEBELUM DAN SESUDAH TRANSFORMASI LOG ===")
print(f"Skewness Harga Asli (IDR)       : {skew_before:.4f}  (Miring kuat ke kanan / Highly Positive Skew)")
print(f"Skewness Log Harga (IDR)        : {skew_after:.4f}   (Mendekati distribusi normal / Moderat)")
print(f"Penurunan Nilai Skewness        : {abs(skew_before - skew_after):.4f} poin\n")

summary_comparison = pd.DataFrame({
    'Parameter Statistik': ['Rata-rata (Mean)', 'Nilai Tengah (Median)', 'Standar Deviasi', 'Minimum', 'Maksimum', 'Kemencengan (Skewness)'],
    'Harga IDR Asli (Rp)': [
        f"{df['price_idr'].mean():,.0f}",
        f"{df['price_idr'].median():,.0f}",
        f"{df['price_idr'].std():,.0f}",
        f"{df['price_idr'].min():,.0f}",
        f"{df['price_idr'].max():,.0f}",
        f"{skew_before:.4f}"
    ],
    'Log(Harga IDR)': [
        f"{df['log_price_idr'].mean():.4f}",
        f"{df['log_price_idr'].median():.4f}",
        f"{df['log_price_idr'].std():.4f}",
        f"{df['log_price_idr'].min():.4f}",
        f"{df['log_price_idr'].max():.4f}",
        f"{skew_after:.4f}"
    ]
})
display(summary_comparison)

=== PERBANDINGAN DISTRIBUSI SEBELUM DAN SESUDAH TRANSFORMASI LOG ===
Skewness Harga Asli (IDR)       : 2.3206  (Miring kuat ke kanan / Highly Positive Skew)
Skewness Log Harga (IDR)        : 0.9747   (Mendekati distribusi normal / Moderat)
Penurunan Nilai Skewness        : 1.3458 poin



,Parameter Statistik,Harga IDR Asli (Rp),Log(Harga IDR)
0,Rata-rata (Mean),"34,392,591",17.2070
1,Nilai Tengah (Median),"26,598,100",17.0964
2,Standar Deviasi,"22,590,400",0.5022
3,Minimum,"11,392,780",16.2485
4,Maksimum,"151,999,810",18.8394
5,Kemencengan (Skewness),2.3206,0.9747


In [13]:
# Visualisasi Perbandingan Distribusi Harga (Jika matplotlib terpasang)
if PLT_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Histogram Harga Asli
    sns.histplot(df['price_idr'] / 1e6, kde=True, ax=axes[0], color='salmon', bins=25)
    axes[0].set_title(f"Distribusi Harga Asli (Juta IDR)\nSkewness = {skew_before:.2f} (Miring Positif)", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Harga Laptop (Juta Rupiah)")
    axes[0].set_ylabel("Frekuensi")
    
    # Plot 2: Histogram Log-Transformed Harga
    sns.histplot(df['log_price_idr'], kde=True, ax=axes[1], color='teal', bins=25)
    axes[1].set_title(f"Distribusi Log-Transformed Harga (log1p)\nSkewness = {skew_after:.2f} (Mendekati Normal)", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("log1p(Harga IDR)")
    axes[1].set_ylabel("Frekuensi")
    
    plt.tight_layout()
    plt.show()
else:
    print("Plot visualisasi dilewati karena matplotlib/seaborn belum terpasang di venv.")

Plot visualisasi dilewati karena matplotlib/seaborn belum terpasang di venv.


---
## 6. Penyusunan Dataset Akhir & Ekspor Data Siap Model Regresi

Kita merangkum seluruh fitur numerik dan ordinal hasil proses preprocessing yang siap digunakan sebagai matriks fitur ($X$) dan target ($y$) untuk pemodelan regresi.

In [14]:
# Memilih kolom fitur inti yang telah dibersihkan
features_final = [
    'name',              # Nama lengkap laptop
    'price_idr',         # Harga asli tersandardisasi dalam Rupiah (Target)
    'log_price_idr',     # Logaritma harga IDR (Target ternormalisasi untuk model regresi linier)
    'ram_gb',            # Kapasitas RAM tersandardisasi (GB)
    'storage_gb',        # Kapasitas Storage tersandardisasi (GB)
    'screen_size_inch',  # Ukuran layar dalam inci
    'refresh_rate_hz',   # Kecepatan refresh rate layar dalam Hz
    'gpu_tier',          # Ordinal tier kartu grafis (Tier 1 s/d 6)
    'cpu_tier',          # Ordinal tier prosesor (Tier 1 s/d 5)
    'VRAM(GB)',          # Kapasitas VRAM kartu grafis
    'specs_score'        # Skor spesifikasi sintetis
]

df_final = df[features_final].copy()

print("=== CUPLIKAN DATASET FINAL HASIL PREPROCESSING ===")
display(df_final.head(10))

# Simpan dataset bersih ke file CSV
output_csv = os.path.join('..', 'raw-dataset', 'dataset_preprocessed_regresi.csv')
try:
    df_final.to_csv(output_csv, index=False)
    print(f"\n✅ Dataset hasil preprocessing berhasil disimpan ke: '{output_csv}'")
except Exception as e:
    try:
        df_final.to_csv('dataset_preprocessed_regresi.csv', index=False)
        print(f"\n✅ Dataset hasil preprocessing berhasil disimpan ke: 'dataset_preprocessed_regresi.csv'")
    except Exception as e2:
        print(f"\nℹ️ File ready in memory (Simpan ke disk: {e2})")

print(f"Total baris: {df_final.shape[0]}, Total fitur: {df_final.shape[1]}.")

=== CUPLIKAN DATASET FINAL HASIL PREPROCESSING ===


,name,price_idr,log_price_idr,ram_gb,storage_gb,screen_size_inch,refresh_rate_hz,gpu_tier,cpu_tier,VRAM(GB),specs_score
0,Lenovo LOQ 83DV007GIN Gaming Laptop,25080000.0,17.037581,16,512,15.6,144.0,3,2,6,59
1,HP Victus 15-fb3125AX Gaming Laptop,15673860.0,16.567505,16,512,15.0,144.0,3,3,6,50
2,HP Victus 15-fa2192TX Gaming Laptop,15008100.0,16.524101,16,512,15.0,144.0,2,2,4,53
3,Lenovo LOQ Essential 83S000JCIN Gaming Laptop,17098100.0,16.654478,16,512,15.6,144.0,2,2,6,64
4,Lenovo LOQ 2023 Gaming Laptop,17099810.0,16.654578,8,512,15.6,144.0,4,2,8,54
5,Acer Aspire 7 A715-79G UN.DWCSI.001 Gaming Laptop,15198100.0,16.536681,16,512,15.6,144.0,2,2,4,53
6,Asus TUF Gaming A15 FA506NCG-HN199W Laptop,18686310.0,16.743302,16,512,15.6,144.0,2,3,4,62
7,HP Omen 16-xf0081AX Gaming Laptop,25839810.0,17.067427,32,1024,16.0,144.0,4,3,8,75
8,Acer Aspire 5 A514-56M 2023 Gaming Laptop,11392780.0,16.248490,8,512,14.0,144.0,1,1,0,46
9,Dell G15-5530 GN5530D83M6002ORB1 Gaming Laptop,21656200.0,16.890802,16,512,15.6,144.0,2,2,6,55



✅ Dataset hasil preprocessing berhasil disimpan ke: '..\raw-dataset\dataset_preprocessed_regresi.csv'
Total baris: 346, Total fitur: 11.


---
## 🎯 Ringkasan & Kesimpulan Preprocessing

| Tahap | Tindakan yang Dilakukan | Hasil Preprocessing |
| :--- | :--- | :--- |
| **Langkah 1: Standardisasi Satuan** | Konversi harga ₹ ke Rupiah via kurs tetap $1\text{ INR} = \text{Rp }190$, RAM & SSD diseragamkan ke Gigabyte (GB). | Kolom `price_idr`, `ram_gb`, dan `storage_gb` bernilai seragam dan mudah diinterpretasikan. |
| **Langkah 2: Ekstraksi Fitur Teks** | Ekstraksi Regex dari teks mentah untuk seri GPU, kelas & generasi CPU, ukuran layar, dan refresh rate. | Terbentuk fitur `gpu_series`, `cpu_class`, `cpu_gen`, `screen_size_inch`, dan `refresh_rate_hz`. |
| **Langkah 3: Pemetaan Tier Performa** | Ordinal encoding bertingkat pada GPU (Tier 1-6) dan CPU (Tier 1-5) via `.map()` & `OrdinalEncoder`. | Fitur kategori hardware berubah menjadi angka bertingkat yang mempertahankan hubungan hierarki komputasi. |
| **Langkah 4: Penanganan Missing & Outlier** | Koreksi anomali storage 1 GB & 8 GB pada ROG Strix Scar ke 1024 GB (1 TB), imputasi refresh rate 144 Hz, evaluasi outlier via IQR. | Data bersih dari input typo/rusak dan bebas dari missing values. |
| **Langkah 5: Transformasi Distribusi** | Logaritma transformasi (`np.log1p`) pada kolom target harga `price_idr`. | Skewness harga turun drastis dari **2.32** menjadi **0.97**, memenuhi asumsi normalitas model regresi. |